# M6 — Image Quality (OpenCV)

Exploration notebook for the M6 image-quality module. Imports the shared logic from
`ml/m6_image_quality/scripts/common.py` and runs the per-image + 8-angle analysis on the
tracked `manual_test/` photos.

**Run from the repo root** (imports are absolute), in the `.venv-m4m5m6` environment:

```
py -3.11 -m venv .venv-m4m5m6
.venv-m4m5m6\Scripts\python -m pip install opencv-python imagehash "numpy<2" pandas pyyaml pytest ipykernel
.venv-m4m5m6\Scripts\python -m ipykernel install --user --name venv-m4m5m6
```

See `README.md` for the CLI equivalents (`scripts/infer.py`, `scripts/smoke_test.py`).

In [ ]:
from pathlib import Path
from pprint import pprint

from ml.m6_image_quality.scripts.common import (
    analyze_image_quality,
    analyze_eight_angle_images,
    compare_image_hashes,
)

## 1. Single-image analysis

In [ ]:
manual_dir = Path("ml/m6_image_quality/manual_test")

sample = next(
    p for p in manual_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
print(f"sample: {sample}")
pprint(analyze_image_quality(str(sample)))

## 2. Duplicate / near-duplicate detection

In [ ]:
images = sorted(
    p for p in manual_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

for i in range(min(3, len(images))):
    for j in range(i + 1, min(4, len(images))):
        dist = compare_image_hashes(str(images[i]), str(images[j]))
        print(f"{images[i].name} <-> {images[j].name}: distance={dist}")

## 3. 8-angle aggregate analysis

In [ ]:
if len(images) == 8:
    aggregate = analyze_eight_angle_images([str(p) for p in images])
    print(f"overall_quality_score : {aggregate['overall_quality_score']}")
    print(f"passed                 : {aggregate['passed_images']}/{aggregate['total_images']}")
    print(f"duplicate pairs        : {len(aggregate['duplicate_or_similar_pairs'])}")
    print("--- per image ---")
    for r in aggregate["image_results"]:
        print(f"  {Path(r['image_path']).name:<14} score={r['quality_score']:>3.0f} issues={r['issues']}")
else:
    print(f"expected 8 manual_test images, found {len(images)}")